In [1]:
pip install azure-identity azure-keyvault-secrets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

VAULT_URL = "https://sqlanalyticsai-kv.vault.azure.net/"

credential = DefaultAzureCredential()

client = SecretClient(
    vault_url=VAULT_URL,
    credential=credential
)

print("Connected Successfully")

Connected Successfully


In [3]:
openai_key = client.get_secret(
    "azure-openai-key"
).value

print(openai_key[:10])

Cz0CuX608G


In [4]:
from IPython.display import display
import pandas as pd
import plotly.express as px


def plot_charts(spark_df):

    # Convert Spark DataFrame to Pandas
    df = spark_df.toPandas()

    if df.empty:
        print("No data returned.")
        return

    print("\n================ RESULT ================\n")
    display(df)

    # Try converting object columns to datetime
    for col in df.columns:
        if df[col].dtype == "object":
            try:
                converted = pd.to_datetime(df[col], errors="raise")
                df[col] = converted
            except:
                pass

    # Detect columns
    datetime_cols = list(df.select_dtypes(include=["datetime64[ns]"]).columns)
    numeric_cols = list(df.select_dtypes(include=["number"]).columns)
    categorical_cols = [
        c for c in df.columns
        if c not in datetime_cols and c not in numeric_cols
    ]

    # -----------------------------
    # CASE 1 : Single value
    # -----------------------------
    if len(df) == 1 and len(df.columns) == 1:
        print("\nSingle aggregated result:")
        display(df)
        return

    # -----------------------------
    # CASE 2 : Time series
    # Example:
    # date | revenue
    # -----------------------------
    if len(datetime_cols) >= 1 and len(numeric_cols) >= 1:

        x = datetime_cols[0]

        for y in numeric_cols:

            fig = px.line(
                df,
                x=x,
                y=y,
                title=f"{y} over {x}",
                markers=True
            )

            fig.update_layout(
                template="plotly_white",
                height=500,
                width=900
            )

            fig.show()

        return

    # -----------------------------
    # CASE 3 : Category vs Number
    # Example:
    # payment_type | avg_fare
    # -----------------------------
    if len(categorical_cols) >= 1 and len(numeric_cols) >= 1:

        x = categorical_cols[0]

        for y in numeric_cols:

            fig = px.bar(
                df,
                x=x,
                y=y,
                text=y,
                title=f"{y} by {x}"
            )

            fig.update_layout(
                template="plotly_white",
                height=500,
                width=900
            )

            fig.show()

        return

    # -----------------------------
    # CASE 4 : Two numeric columns
    # Example:
    # distance | fare
    # -----------------------------
    if len(numeric_cols) >= 2:

        fig = px.scatter(
            df,
            x=numeric_cols[0],
            y=numeric_cols[1],
            title=f"{numeric_cols[1]} vs {numeric_cols[0]}"
        )

        fig.update_layout(
            template="plotly_white",
            height=500,
            width=900
        )

        fig.show()

        return

    # -----------------------------
    # CASE 5 : Fallback
    # -----------------------------
    print("\nCould not determine a suitable chart.")
    display(df)

In [9]:
def run_sql_command_on_spark(sql_command):
    spark = SparkSession.builder \
        .appName("DeltaTableReader") \
        .master("local[*]") \
        .getOrCreate()

    delta_table_path = r"F:\Data_Engineering\SQLAnalyticsAI\AIProjects\yellow_tripdata_2024-07.parquet"

    df = spark.read.format("parquet").load(delta_table_path)

    df.createOrReplaceTempView("taxi_trips")

    result = spark.sql(sql_command)

    plot_charts(result)

In [10]:
is_debugging_enabled = False

In [11]:
import os  
import base64
import json
from openai import AzureOpenAI  

endpoint = os.getenv("ENDPOINT_URL", "https://tbomb-mq56qohq-eastus2.openai.azure.com/")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4o")
search_endpoint = os.getenv("SEARCH_ENDPOINT", "https://newaisearchtejas.search.windows.net/")
search_key = client.get_secret("azure-search-key").value
subscription_key = client.get_secret("azure-openai-key").value

print("Block : call_llm")

def call_llm(user_input):
    # Initialize Azure OpenAI client with key-based authentication    
    client = AzureOpenAI(  
        azure_endpoint=endpoint,  
        api_key=subscription_key,  
        api_version="2024-05-01-preview",  
    )
    prompt_content = "You are an AI assistant specialized in transforming natural language into SQL queries.The dataset includes embedding vectors representing predefined query patterns. Use these embeddings to adapt the generated SQL queries, aligning them with patterns found in the embedding dataset attached to the query. Return only the SQL output without explanations or comments"    #Prepare the chat prompt 
    chat_prompt = [
        {
            "role": "system",
            "content": f"{prompt_content}"
        },
        {
            "role": "user",
            "content": f"{user_input}"
        }
    ]
        
    # Include speech result if speech is enabled  
    messages = chat_prompt  
        
    # Generate the completion  
    completion = client.chat.completions.create(  
        model=deployment,  
        messages=messages,  
        max_tokens=800,  
        temperature=0.67,  
        top_p=0.95,  
        frequency_penalty=0,  
        presence_penalty=0,  
        stop=None,  
        stream=False,
        extra_body={
          "data_sources": [{
              "type": "azure_search",
              "parameters": {
                "filter": None,
                "endpoint": f"{search_endpoint}",
                 "index_name": "calm-clock-5zdftp8st5",
                "semantic_configuration": "azureml-default",
                "authentication": {
                  "type": "api_key",
                  "key": f"{search_key}"
                },
                "embedding_dependency": {
                  "type": "endpoint",
                  "endpoint": "https://tbomb-mq56qohq-eastus2.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2023-07-01-preview",
                  "authentication": {
                    "type": "api_key",
                    "key": f"{subscription_key}"
                  }
                },
                "query_type": "simple",
                "in_scope": True,
                "role_information": f"{prompt_content}",
                "strictness": 3,
                "top_n_documents": 5
              }
            }]
        }
    )
    
    res = completion.to_json()
    res = json.loads(res)
    content = res["choices"][0]["message"]["content"]
    print("Key response from LLM " + content)

    if "SELECT" in content:
        # Remove ```sql and ``` from the content
        clean_sql = content.replace("```sql", "").replace("```", "").strip()
        if is_debugging_enabled:
            print("Cleaned up SQL command:\n" + clean_sql)
        run_sql_command_on_spark(clean_sql)

Block : call_llm


In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Text input widget
text_input = widgets.Textarea(
    placeholder='Ask your question here...',
    layout=widgets.Layout(width='80%', height='50px')
)

# def call_llm(text):
#     print(f"I am text! Here's the input:\n{text}")
        
# Callback function to handle button click
def button_callback(b):
    with output:
        clear_output(wait=True)
        text = text_input.value
        call_llm(text)
        
        
# Button to trigger the action
button = widgets.Button(description='Search')

# Output widget to show results
output = widgets.Output()

# Attach the function to the button click event
button.on_click(button_callback)

# Display the widgets
display(text_input, button, output)


Textarea(value='', layout=Layout(height='50px', width='80%'), placeholder='Ask your question here...')

Button(description='Search', style=ButtonStyle())

Output()